# Project 2 — CNN Image Classification & Transfer Learning

Public portfolio version of coursework for the Image and Video Analysis course at NTUA. The original staff handout is intentionally not distributed.

This notebook compares classical CNN architectures with a custom model on a CIFAR-100 subset, then explores regularization and transfer learning.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score


## Dataset

The coursework used a deterministic class subset of CIFAR-100 selected from a team-specific seed. Images are normalized to `[0, 1]` and labels are remapped to the selected classes.

In [ ]:
team_seed = 4
num_classes = 10

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data(label_mode='fine')
y_train = y_train.squeeze()
y_test = y_test.squeeze()

rng = np.random.default_rng(team_seed)
selected_classes = np.sort(rng.choice(100, size=num_classes, replace=False))
class_to_idx = {c: i for i, c in enumerate(selected_classes)}

train_mask = np.isin(y_train, selected_classes)
test_mask = np.isin(y_test, selected_classes)
x_train, y_train = x_train[train_mask] / 255.0, y_train[train_mask]
x_test, y_test = x_test[test_mask] / 255.0, y_test[test_mask]
y_train = np.array([class_to_idx[y] for y in y_train])
y_test = np.array([class_to_idx[y] for y in y_test])

split = int(0.8 * len(x_train))
x_val, y_val = x_train[split:], y_train[split:]
x_train, y_train = x_train[:split], y_train[:split]

print('Selected CIFAR-100 classes:', selected_classes)
print(x_train.shape, x_val.shape, x_test.shape)


## Architectures

The project compares LeNet, AlexNet, VGG-style and custom CNN architectures under the same classification task.

In [ ]:
def build_lenet(input_shape=(32,32,3), num_classes=10):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(6, 5, activation='tanh', padding='same'),
        layers.AveragePooling2D(),
        layers.Conv2D(16, 5, activation='tanh'),
        layers.AveragePooling2D(),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84, activation='tanh'),
        layers.Dense(num_classes, activation='softmax')
    ])

def build_alexnet(input_shape=(32,32,3), num_classes=10):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(64, 5, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

def build_vgg(input_shape=(32,32,3), num_classes=10):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

def build_mycnn(input_shape=(32,32,3), num_classes=10, dropout=0.0):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Dropout(dropout),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Dropout(dropout),
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation='softmax')
    ])


## Common training/evaluation helpers

In [ ]:
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def macro_f1(model, x, y):
    pred = model.predict(x, verbose=0).argmax(axis=1)
    return f1_score(y, pred, average='macro')

def train_model(model, epochs=50, batch_size=64, augmentation=None):
    if augmentation is None:
        x = x_train
    else:
        x = augmentation(x_train, training=True)
    history = model.fit(
        x, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )
    return history


## Baseline experiment

The original coursework trained LeNet, AlexNet, VGG and MyCNN with a common protocol and compared their F1-scores. The custom CNN achieved the strongest baseline result among the four models.

In [ ]:
builders = {
    'LeNet': build_lenet,
    'AlexNet': build_alexnet,
    'VGG': build_vgg,
    'MyCNN': build_mycnn,
}

# Example usage (training all models can take time):
# results = {}
# for name, builder in builders.items():
#     model = compile_model(builder(num_classes=num_classes))
#     history = train_model(model, epochs=50, batch_size=64)
#     results[name] = macro_f1(model, x_test, y_test)
# print(results)


## Regularization: dropout and data augmentation

The custom CNN was retrained with dropout, data augmentation, and their combination to reduce overfitting. The combined configuration produced the best reported test-set F1-score in the original notebook.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomRotation(0.05),
])

regularized_model = compile_model(build_mycnn(num_classes=num_classes, dropout=0.3))

# More scalable pipeline than materializing augmented arrays:
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(len(x_train)).batch(64)
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

# regularized_model.fit(train_ds, validation_data=(x_val, y_val), epochs=60)


### Reported regularization results

From the original coursework notebook:

- Dropout: **F1 = 0.7136**
- Data augmentation: **F1 = 0.7439**
- Dropout + data augmentation: **F1 = 0.7713**

The combined strategy was retained for the later comparison with transfer learning.

## Transfer learning

VGG19 and EfficientNetB0 are used as pretrained feature extractors. The project studies both frozen-backbone training and partial fine-tuning.

In [ ]:
def build_transfer_model(backbone_name, num_classes=10, trainable_fraction=0.0):
    if backbone_name == 'VGG19':
        base = keras.applications.VGG19(include_top=False, weights='imagenet', input_shape=(224,224,3))
        preprocess = keras.applications.vgg19.preprocess_input
    elif backbone_name == 'EfficientNetB0':
        base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224,224,3))
        preprocess = keras.applications.efficientnet.preprocess_input
    else:
        raise ValueError('Unknown backbone')

    base.trainable = True
    cutoff = int(len(base.layers) * (1 - trainable_fraction))
    for layer in base.layers[:cutoff]:
        layer.trainable = False

    inputs = keras.Input((32,32,3))
    x = layers.Resizing(224,224)(inputs)
    x = preprocess(x * 255.0)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs)

# Frozen backbone:
# vgg19_frozen = compile_model(build_transfer_model('VGG19', num_classes, 0.0), lr=1e-3)
# efficientnet_frozen = compile_model(build_transfer_model('EfficientNetB0', num_classes, 0.0), lr=1e-3)

# Partial fine-tuning, typically with a lower learning rate:
# vgg19_ft = compile_model(build_transfer_model('VGG19', num_classes, 0.2), lr=1e-5)
# efficientnet_ft = compile_model(build_transfer_model('EfficientNetB0', num_classes, 0.2), lr=1e-5)


## Transfer-learning observations

The original experiments found that the pretrained models did not outperform the best custom CNN configuration on this small 32×32 task. Reported fine-tuning test F1-scores included approximately **0.5925 for VGG19** and **0.5459 for EfficientNetB0**.

Possible reasons discussed in the coursework included the mismatch between CIFAR image resolution and the much larger ImageNet input scale, the complexity of the pretrained networks relative to the task, and the need for careful fine-tuning schedules.

## Scalable input pipelines

For larger datasets, loading every image into RAM becomes inefficient. TensorFlow's `tf.data` API and TFRecord serialization support streaming, parallel preprocessing, caching and prefetching, making them preferable for larger-scale experiments.

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
dataset = dataset.shuffle(len(x_train)).batch(64).prefetch(tf.data.AUTOTUNE)
dataset


## Summary

This project combines architecture comparison, regularization experiments and transfer learning on an image-classification problem. The strongest reported result came from the custom CNN with both dropout and data augmentation.